# Template Management Notebook

This notebook helps manage website templates by:
1. Creating a directory listing of templates
2. Saving the list to a Markdown file
3. Updating base.html and server.py to use the templates

Let's start by importing the required libraries.

In [ ]:
# Import Required Libraries
import os
import pathlib
from datetime import datetime
import re
import shutil

# List Files in Directory

Let's list all the files in the 'src/web/templates' directory. We'll use the `os` module for this operation and organize the files by their type.

In [ ]:
# Define the templates directory path
templates_dir = 'src/web/templates'

# Check if the directory exists
if not os.path.exists(templates_dir):
    print(f"Directory {templates_dir} does not exist. Creating it...")
    os.makedirs(templates_dir, exist_ok=True)

# Get all files in the templates directory
template_files = []
for root, dirs, files in os.walk(templates_dir):
    for file in files:
        file_path = os.path.join(root, file)
        relative_path = os.path.relpath(file_path, templates_dir)
        template_files.append({
            'name': file,
            'path': relative_path,
            'full_path': file_path,
            'type': os.path.splitext(file)[1],
            'modified': datetime.fromtimestamp(os.path.getmtime(file_path)).strftime('%Y-%m-%d %H:%M:%S')
        })

# Display the template files
print(f"Found {len(template_files)} template files:")
for i, template in enumerate(template_files, 1):
    print(f"{i}. {template['path']} ({template['type']}) - Last modified: {template['modified']}")

# Write Directory Listing to Markdown

Now that we have the list of template files, let's write this information to a Markdown file. We'll organize the files by type and include metadata like modification dates.

In [ ]:
# Create a markdown file with the template listing
markdown_file_path = 'src/web/templates_listing.md'

with open(markdown_file_path, 'w') as md_file:
    md_file.write("# Website Templates Listing\n\n")
    md_file.write(f"*Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
    
    # Group files by extension
    file_types = {}
    for template in template_files:
        file_type = template['type'].lstrip('.') or 'other'
        if file_type not in file_types:
            file_types[file_type] = []
        file_types[file_type].append(template)
    
    # Write each file type section
    for file_type, files in file_types.items():
        md_file.write(f"## {file_type.upper()} Files\n\n")
        md_file.write("| File | Path | Last Modified |\n")
        md_file.write("|------|------|---------------|\n")
        
        for file in files:
            md_file.write(f"| {file['name']} | {file['path']} | {file['modified']} |\n")
        
        md_file.write("\n")

print(f"Template listing saved to {markdown_file_path}")

# Display the content of the markdown file
with open(markdown_file_path, 'r') as md_file:
    print(md_file.read())

# Update base.html

Now let's update the `base.html` template to include a dynamic sidebar with links to all the available templates. We'll read the existing base.html file, modify it, and save the changes.

In [ ]:
# Define the base.html path
base_html_path = 'src/web/templates/base.html'

# Create a backup of the original file
if os.path.exists(base_html_path):
    backup_path = f"{base_html_path}.backup.{datetime.now().strftime('%Y%m%d%H%M%S')}"
    shutil.copy2(base_html_path, backup_path)
    print(f"Created backup of base.html at {backup_path}")
else:
    # Create a basic base.html if it doesn't exist
    os.makedirs(os.path.dirname(base_html_path), exist_ok=True)
    with open(base_html_path, 'w') as html_file:
        html_file.write("""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}ImpressionCore{% endblock %}</title>
    <style>
        body { display: flex; min-height: 100vh; margin: 0; }
        .sidebar { width: 250px; background: #f0f0f0; padding: 1rem; }
        .content { flex: 1; padding: 1rem; }
    </style>
</head>
<body>
    <div class="sidebar">
        <h2>Navigation</h2>
        <ul>
            <li><a href="/">Home</a></li>
            <!-- Template links will be inserted here -->
        </ul>
    </div>
    <div class="content">
        {% block content %}{% endblock %}
    </div>
</body>
</html>
""")
        print("Created new base.html template")

# Read the base.html file
with open(base_html_path, 'r') as html_file:
    html_content = html_file.read()

# Generate the template links HTML
template_links = "\n            <!-- Auto-generated template links -->\n"
html_templates = [t for t in template_files if t['type'] == '.html' and 'base.html' not in t['name']]

for template in html_templates:
    # Extract route name from filename (remove extension and special characters)
    route_name = os.path.splitext(template['name'])[0]
    display_name = route_name.replace('_', ' ').title()
    template_links += f'            <li><a href="/{route_name}">{display_name}</a></li>\n'

template_links += '            <!-- End of auto-generated links -->'

# Update the content with the new template links
if '<!-- Template links will be inserted here -->' in html_content:
    html_content = html_content.replace('<!-- Template links will be inserted here -->', template_links)
else:
    # Find the navigation list and update it
    pattern = r'<ul>[\s\S]*?<\/ul>'
    if re.search(pattern, html_content):
        old_list = re.search(pattern, html_content).group(0)
        new_list = '<ul>\n            <li><a href="/">Home</a></li>\n' + template_links + '\n        </ul>'
        html_content = html_content.replace(old_list, new_list)

# Write the updated content back to base.html
with open(base_html_path, 'w') as html_file:
    html_file.write(html_content)

print(f"Updated {base_html_path} with template links")

# Update server.py

Finally, let's update the `server.py` file to dynamically serve all the templates we've discovered. This will create routes for each template and ensure they're accessible through the web server.

In [ ]:
# Define the server.py path
server_py_path = 'src/web/server.py'

# Create a backup of the original file
if os.path.exists(server_py_path):
    backup_path = f"{server_py_path}.backup.{datetime.now().strftime('%Y%m%d%H%M%S')}"
    shutil.copy2(server_py_path, backup_path)
    print(f"Created backup of server.py at {backup_path}")
else:
    # Create the directory if it doesn't exist
    os.makedirs(os.path.dirname(server_py_path), exist_ok=True)
    
    # Create a basic server.py if it doesn't exist
    with open(server_py_path, 'w') as py_file:
        py_file.write("""from flask import Flask, render_template, redirect, url_for

app = Flask(__name__)

@app.route('/')
def home():
    return render_template('index.html')

# Additional routes will be added here

if __name__ == '__main__':
    app.run(debug=True)
""")
    print("Created new server.py file")

# Read the server.py file
with open(server_py_path, 'r') as py_file:
    server_content = py_file.read()

# Generate the dynamic route code
dynamic_routes = "\n# Dynamic routes for templates - Auto-generated\n"
html_templates = [t for t in template_files if t['type'] == '.html' and 'base.html' not in t['name']]

for template in html_templates:
    # Extract route name from filename (remove extension)
    template_name = os.path.splitext(template['name'])[0]
    # Skip index.html as it's already handled by the home route
    if template_name == 'index':
        continue
    
    route_path = template_name.replace('_', '-')
    dynamic_routes += f"""
@app.route('/{route_path}')
def {template_name.replace('-', '_')}():
    return render_template('{template['path']}')
"""

dynamic_routes += "\n# End of auto-generated routes\n"

# Update the content with the new routes
if '# Additional routes will be added here' in server_content:
    server_content = server_content.replace('# Additional routes will be added here', dynamic_routes)
else:
    # Add routes before the if __name__ == '__main__' line
    pattern = r"if __name__ == '__main__':"
    if pattern in server_content:
        server_content = server_content.replace(pattern, dynamic_routes + "\nif __name__ == '__main__':")
    else:
        # Just append to the end if the pattern isn't found
        server_content += dynamic_routes

# Write the updated content back to server.py
with open(server_py_path, 'w') as py_file:
    py_file.write(server_content)

print(f"Updated {server_py_path} with dynamic routes for templates")

# Summary

In this notebook, we have:

1. Listed all files in the `src/web/templates` directory
2. Created a Markdown file (`templates_listing.md`) with an organized list of all templates
3. Updated the `base.html` file to include navigation links to all available templates
4. Modified `server.py` to dynamically create routes for each template

This automation ensures that whenever you add a new template, you can run this notebook to update all the necessary components without manual intervention.

## Next Steps

- Add more metadata to the templates listing (e.g., template description, author)
- Implement template categories for better organization
- Create a web interface for template management